<a href="https://colab.research.google.com/github/mikske/Agentic_YouTube_Search_for_Game-movies/blob/main/agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Проект: Диалоговый агент для поиска игрофильмов без комментариев на YouTube

Я решила попробовать решить вопрос, которым самой часто приходится заниматься и, как ясно из названия проекта, это поиск игрофильмов на сервисе YouTube (я смотрю их как реальные фильмы, когда понимаю, что геймплей мне не заходит, а понять, что происходит и быть в теме все равно хочется).

Из этого вытекает наша цель: создание агента, который принимает запрос пользователя на естественном языке, извлекает название игры и предпочтения, обращается к YouTube API, находит релевантные видео и возвращает ответ в диалоговой форме.

Т.е. идея такая, если смотреть по пунктам:
> 1. Принимаем запрос пользователя;
> 2. Интерпретируем его;
> 3. Плаируем действия;
> 4. Используем внешний инструмент;
> 5. Оцениваем и ранжируем результат;
> 6. Возвращаем осмысленный ответ.

In [77]:
import json
from google.colab import drive

drive.mount('/content/drive')

path = "/content/drive/MyDrive/Colab Notebooks/agent.ipynb"

with open("/content/drive/MyDrive/Colab Notebooks/agent.ipynb", "r", encoding="utf-8") as f:
    nb = json.load(f)

if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

with open("agent_fixed.ipynb", "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Начинаем с базы --- импорты.

In [51]:
!pip install -q isodate pandas
!pip install -q sentence-transformers

In [52]:
import re
import math
import time
import requests
import pandas as pd
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Dict, Any
from urllib.parse import quote
from isodate import parse_duration
import json

from sentence_transformers import SentenceTransformer

#для Colab Secrets
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

from huggingface_hub import InferenceClient

Все установилось и подтянулось, так что дальше будем читать т.н. API-ключ.

API-ключ, как положено, хранится в Secrets, добыла на просторах Google cloud.

In [53]:
def get_secret(name: str, default: Optional[str] = None) -> Optional[str]:
    if IN_COLAB:
        try:
            return userdata.get(name)
        except Exception:
            return default
    return default

YOUTUBE_API_KEY = get_secret("YOUTUBE_API_KEY")
HF_TOKEN = get_secret("HF_TOKEN")

if not YOUTUBE_API_KEY:
    raise ValueError("Не найден YOUTUBE_API_KEY в Colab Secrets.")

if not HF_TOKEN:
    raise ValueError("Не найден HF_TOKEN в Colab Secrets.")

print("Ключи загружены.")

Ключи загружены.


Инициализируем HF-client.

In [54]:
hf_client = InferenceClient(api_key=HF_TOKEN)
HF_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

Теперь быстренький тестик на всякий случай.

In [55]:
test_response = hf_client.chat_completion(
    model=HF_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Reply with only: ok"}
    ],
    max_tokens=10,
)

print(test_response.choices[0].message.content)

ok


Далее сущности нашего проектика.

In [56]:
@dataclass
class UserIntent:
    raw_query: str
    game_title: str
    wants_no_commentary: bool = True
    wants_full_game: bool = True
    prefers_russian: bool = False
    preferred_terms: List[str] = field(default_factory=list)

@dataclass
class VideoCandidate:
    video_id: str
    title: str
    description: str
    channel_title: str
    published_at: str
    duration_seconds: int = 0
    view_count: int = 0
    url: str = ""

@dataclass
class ScoredVideo:
    video: VideoCandidate
    score: int
    reasons: List[str] = field(default_factory=list)

Далее прописываю некоторые вспомогательные функции + нагло украденную функцию для преобразования ISO 8601 duration в секунды.

In [57]:
def clean_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def contains_cyrillic(text: str) -> bool:
    return bool(re.search(r"[А-Яа-яЁё]", text))

def iso8601_to_seconds(duration_str: str) -> int:
    pattern = r"P(?:(\d+)D)?T?(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?"
    match = re.match(pattern, duration_str)
    if not match:
        return 0

    days = int(match.group(1)) if match.group(1) else 0
    hours = int(match.group(2)) if match.group(2) else 0
    minutes = int(match.group(3)) if match.group(3) else 0
    seconds = int(match.group(4)) if match.group(4) else 0

    return days * 86400 + hours * 3600 + minutes * 60 + seconds

def format_seconds(seconds: int) -> str:
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h}h {m}m {s}s"
    return f"{m}m {s}s"

Далее делаем наш LLM-парсер через HF.

In [58]:
INTENT_PROMPT = """
You extract structured intent for a YouTube game-video search agent.

Return ONLY valid JSON in this exact schema:
{
  "game_title": "string",
  "wants_no_commentary": true,
  "wants_full_game": true,
  "prefers_russian": false,
  "preferred_terms": ["game movie", "walkthrough"]
}

Rules:
- game_title: only the game title, no extra words.
- wants_no_commentary: true if user wants no commentary OR if unspecified.
- wants_full_game: true if user wants a full playthrough, game movie, walkthrough, or if unspecified.
- prefers_russian: true if user writes in Russian or asks for Russian results.
- preferred_terms: short list of useful search terms, максимум 5.
Return JSON only.
"""

def llm_parse_user_intent(user_query: str) -> UserIntent:
    response = hf_client.chat_completion(
        model=HF_MODEL,
        messages=[
            {"role": "system", "content": INTENT_PROMPT},
            {"role": "user", "content": user_query}
        ],
        max_tokens=300,
        temperature=0.2,
    )

    text = response.choices[0].message.content.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    data = json.loads(text)

    return UserIntent(
        raw_query=user_query,
        game_title=clean_spaces(data.get("game_title", user_query)),
        wants_no_commentary=bool(data.get("wants_no_commentary", True)),
        wants_full_game=bool(data.get("wants_full_game", True)),
        prefers_russian=bool(data.get("prefers_russian", contains_cyrillic(user_query))),
        preferred_terms=data.get("preferred_terms", []) or []
    )

Здесь генерируем поисковые запросы. Идея такая: чтобы работало и на английском, и на русском языках, потому что я запросы пишу на одном из этих языков в зависимости от того, какая раскладка включена.

In [59]:
def build_search_queries(intent: UserIntent) -> List[str]:
    game = intent.game_title
    queries = []

    if intent.wants_no_commentary and intent.wants_full_game:
        queries.extend([
            f"{game} full game no commentary",
            f"{game} game movie no commentary",
            f"{game} walkthrough no commentary",
            f"{game} игрофильм без комментариев",
            f"{game} прохождение без комментариев",
        ])
    elif intent.wants_no_commentary:
        queries.extend([
            f"{game} no commentary",
            f"{game} без комментариев",
        ])
    else:
        queries.extend([
            game,
            f"{game} full game",
            f"{game} walkthrough",
        ])

    for term in intent.preferred_terms:
        term = clean_spaces(term)
        if term:
            queries.append(f"{game} {term}")

    if intent.prefers_russian:
        queries.extend([
            f"{game} игрофильм",
            f"{game} прохождение",
        ])
    else:
        queries.extend([
            f"{game} no commentary",
            f"{game} full walkthrough",
        ])

    unique = []
    seen = set()
    for q in queries:
        if q not in seen:
            unique.append(q)
            seen.add(q)

    return unique[:8]

Тут у нас YouTube helper.

In [60]:
YOUTUBE_SEARCH_URL = "https://www.googleapis.com/youtube/v3/search"
YOUTUBE_VIDEOS_URL = "https://www.googleapis.com/youtube/v3/videos"

def youtube_get(url: str, params: Dict[str, Any]) -> Dict[str, Any]:
    params = dict(params)
    params["key"] = YOUTUBE_API_KEY

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()

Далее непосредственно поиск.

In [61]:
def search_youtube_videos(
    query: str,
    max_results: int = 5,
    page_token: Optional[str] = None,
    relevance_language: Optional[str] = None,
    region_code: Optional[str] = None
) -> Dict[str, Any]:
    params = {
        "part": "snippet",
        "type": "video",
        "q": query,
        "maxResults": max_results,
        "safeSearch": "moderate",
    }

    if page_token:
        params["pageToken"] = page_token
    if relevance_language:
        params["relevanceLanguage"] = relevance_language
    if region_code:
        params["regionCode"] = region_code

    return youtube_get(YOUTUBE_SEARCH_URL, params)

Парсинг результатов.

In [62]:
def parse_search_items(search_response: Dict[str, Any]) -> List[VideoCandidate]:
    items = search_response.get("items", [])
    candidates = []

    for item in items:
        video_id = item.get("id", {}).get("videoId")
        snippet = item.get("snippet", {})

        if not video_id:
            continue

        candidates.append(
            VideoCandidate(
                video_id=video_id,
                title=snippet.get("title", ""),
                description=snippet.get("description", ""),
                channel_title=snippet.get("channelTitle", ""),
                published_at=snippet.get("publishedAt", ""),
                url=f"https://www.youtube.com/watch?v={video_id}"
            )
        )

    return candidates

Здесь у нас детали видео.

In [63]:
def fetch_video_details(video_ids: List[str]) -> Dict[str, Dict[str, Any]]:
    if not video_ids:
        return {}

    params = {
        "part": "contentDetails,statistics,snippet",
        "id": ",".join(video_ids)
    }

    data = youtube_get(YOUTUBE_VIDEOS_URL, params)
    items = data.get("items", [])

    details = {}
    for item in items:
        vid = item.get("id")
        content_details = item.get("contentDetails", {})
        statistics = item.get("statistics", {})
        snippet = item.get("snippet", {})

        details[vid] = {
            "duration_seconds": iso8601_to_seconds(content_details.get("duration", "")),
            "view_count": int(statistics.get("viewCount", 0)) if statistics.get("viewCount") else 0,
            "tags": snippet.get("tags", [])
        }

    return details

И сбор кандидатов!

In [64]:
def collect_candidates(
    queries: List[str],
    max_per_query: int = 4,
    relevance_language: Optional[str] = None,
    region_code: Optional[str] = None,
    sleep_between_requests: float = 0.2
) -> List[VideoCandidate]:
    all_candidates = []

    for q in queries:
        print(f"Ищу: {q}")
        response = search_youtube_videos(
            query=q,
            max_results=max_per_query,
            relevance_language=relevance_language,
            region_code=region_code
        )
        candidates = parse_search_items(response)
        all_candidates.extend(candidates)
        time.sleep(sleep_between_requests)

    dedup = {}
    for c in all_candidates:
        dedup[c.video_id] = c

    unique_candidates = list(dedup.values())
    details = fetch_video_details([c.video_id for c in unique_candidates])

    for c in unique_candidates:
        if c.video_id in details:
            c.duration_seconds = details[c.video_id]["duration_seconds"]
            c.view_count = details[c.video_id]["view_count"]

    return unique_candidates

Дальше обязательно скоринг система.

In [65]:
POSITIVE_KEYWORDS = {
    "no commentary": 5,
    "without commentary": 5,
    "full game": 4,
    "game movie": 4,
    "walkthrough": 3,
    "all cutscenes": 3,
    "игрофильм": 4,
    "без комментариев": 5,
    "прохождение без комментариев": 5,
    "full walkthrough": 3,
    "movie edition": 2,
    "cinematic": 2
}

NEGATIVE_KEYWORDS = {
    "commentary": -6,
    "with commentary": -7,
    "let's play": -6,
    "lets play": -6,
    "stream": -7,
    "livestream": -7,
    "reaction": -6,
    "review": -5,
    "обзор": -5,
    "стрим": -7,
    "летсплей": -6,
    "с комментариями": -7,
    "funny moments": -5,
    "podcast": -5
}

def score_video(video: VideoCandidate) -> ScoredVideo:
    text = f"{video.title} {video.description}".lower()
    score = 0
    reasons = []

    for kw, points in POSITIVE_KEYWORDS.items():
        if kw in text:
            score += points
            reasons.append(f"+{points}: '{kw}'")

    for kw, points in NEGATIVE_KEYWORDS.items():
        if kw in text:
            score += points
            reasons.append(f"{points}: '{kw}'")

    if video.duration_seconds >= 3 * 3600:
        score += 4
        reasons.append("+4: длинное видео > 3 часов")
    elif video.duration_seconds >= 90 * 60:
        score += 2
        reasons.append("+2: длинное видео > 90 минут")
    elif 0 < video.duration_seconds < 20 * 60:
        score -= 4
        reasons.append("-4: слишком короткое видео")

    if video.view_count >= 500_000:
        score += 2
        reasons.append("+2: много просмотров")
    elif video.view_count >= 50_000:
        score += 1
        reasons.append("+1: заметное число просмотров")

    if "trailer" in text or "тизер" in text:
        score -= 6
        reasons.append("-6: похоже на трейлер")

    return ScoredVideo(video=video, score=score, reasons=reasons)

Ранжируемс наши результаты.

In [66]:
def rank_videos(candidates: List[VideoCandidate]) -> List[ScoredVideo]:
    ranked = [score_video(c) for c in candidates]
    ranked.sort(key=lambda x: x.score, reverse=True)
    return ranked

И генерируем LLM ответ через HF.

In [67]:
def llm_build_answer(user_query: str, intent: UserIntent, ranked_videos: List[ScoredVideo], top_k: int = 5) -> str:
    top = ranked_videos[:top_k]

    compact_results = []
    for item in top:
        compact_results.append({
            "title": item.video.title,
            "channel": item.video.channel_title,
            "duration_seconds": item.video.duration_seconds,
            "view_count": item.video.view_count,
            "score": item.score,
            "url": item.video.url,
            "reasons": item.reasons[:5],
        })

    prompt = f"""
Ты — YouTube-агент для поиска игрофильмов без комментариев.

Запрос пользователя:
{user_query}

Извлеченный intent:
{json.dumps(intent.__dict__, ensure_ascii=False, indent=2)}

Топ результатов:
{json.dumps(compact_results, ensure_ascii=False, indent=2)}

Сформируй краткий ответ на русском языке:
- скажи, что найдено;
- выдели лучший результат;
- кратко объясни, почему он подходит;
- перечисли еще 2-3 варианта;
- если результаты неидеальны, честно скажи это.
"""

    response = hf_client.chat_completion(
        model=HF_MODEL,
        messages=[
            {"role": "system", "content": "Ты полезный и краткий ассистент."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=500,
        temperature=0.4,
    )

    return response.choices[0].message.content.strip()

Вот тут у нас мейн агента.

In [68]:
def run_agent(
    user_query: str,
    max_per_query: int = 4,
    top_k: int = 5,
    relevance_language: Optional[str] = None,
    region_code: Optional[str] = None
) -> Dict[str, Any]:
    print("Шаг 1. Интерпретирую запрос через LLM...")
    intent = llm_parse_user_intent(user_query)
    print("Intent:", intent)

    print("\nШаг 2. Генерирую поисковые запросы...")
    queries = build_search_queries(intent)
    for q in queries:
        print(" -", q)

    print("\nШаг 3. Ищу кандидатов на YouTube...")
    candidates = collect_candidates(
        queries=queries,
        max_per_query=max_per_query,
        relevance_language=relevance_language,
        region_code=region_code
    )
    print(f"Найдено уникальных кандидатов: {len(candidates)}")

    print("\nШаг 4. Ранжирую результаты...")
    ranked = rank_videos(candidates)

    print("\nШаг 5. Формирую финальный ответ через LLM...")
    answer = llm_build_answer(user_query, intent, ranked, top_k=top_k)

    return {
        "intent": intent,
        "queries": queries,
        "candidates": candidates,
        "ranked": ranked,
        "answer": answer
    }

Пробуем небольшой тестик.

In [69]:
result = run_agent("Найди игрофильм по Warhammer 40k: Space Marine без комментариев")
print()
print(result["answer"])

Шаг 1. Интерпретирую запрос через LLM...
Intent: UserIntent(raw_query='Найди игрофильм по Warhammer 40k: Space Marine без комментариев', game_title='Warhammer 40k: Space Marine', wants_no_commentary=True, wants_full_game=True, prefers_russian=False, preferred_terms=['игрофильм', 'без комментариев'])

Шаг 2. Генерирую поисковые запросы...
 - Warhammer 40k: Space Marine full game no commentary
 - Warhammer 40k: Space Marine game movie no commentary
 - Warhammer 40k: Space Marine walkthrough no commentary
 - Warhammer 40k: Space Marine игрофильм без комментариев
 - Warhammer 40k: Space Marine прохождение без комментариев
 - Warhammer 40k: Space Marine игрофильм
 - Warhammer 40k: Space Marine без комментариев
 - Warhammer 40k: Space Marine no commentary

Шаг 3. Ищу кандидатов на YouTube...
Ищу: Warhammer 40k: Space Marine full game no commentary
Ищу: Warhammer 40k: Space Marine game movie no commentary
Ищу: Warhammer 40k: Space Marine walkthrough no commentary
Ищу: Warhammer 40k: Space Mar

Ну как бы нормально, но было бы славно еще и ссылки на видео как-то получить, а не ходить в ютуб самостоятельно.

In [70]:
from IPython.display import display, HTML
import pandas as pd

def show_clickable_table(ranked, top_k=5):
    rows = []
    for item in ranked[:top_k]:
        v = item.video
        rows.append({
            "Название": v.title,
            "Канал": v.channel_title,
            "Длительность": format_seconds(v.duration_seconds),
            "Score": item.score,
            "Видео": f'<a href="{v.url}" target="_blank">Открыть</a>'
        })

    df = pd.DataFrame(rows)
    display(HTML(df.to_html(escape=False, index=False)))

Тестируем.

In [71]:
result = run_agent("Найди игрофильм по Warhammer 40k: Space Marine без комментариев")

print(result["answer"])

show_clickable_table(result["ranked"], top_k=5)

Шаг 1. Интерпретирую запрос через LLM...
Intent: UserIntent(raw_query='Найди игрофильм по Warhammer 40k: Space Marine без комментариев', game_title='Warhammer 40k: Space Marine', wants_no_commentary=True, wants_full_game=True, prefers_russian=False, preferred_terms=['игрофильм', 'без комментариев'])

Шаг 2. Генерирую поисковые запросы...
 - Warhammer 40k: Space Marine full game no commentary
 - Warhammer 40k: Space Marine game movie no commentary
 - Warhammer 40k: Space Marine walkthrough no commentary
 - Warhammer 40k: Space Marine игрофильм без комментариев
 - Warhammer 40k: Space Marine прохождение без комментариев
 - Warhammer 40k: Space Marine игрофильм
 - Warhammer 40k: Space Marine без комментариев
 - Warhammer 40k: Space Marine no commentary

Шаг 3. Ищу кандидатов на YouTube...
Ищу: Warhammer 40k: Space Marine full game no commentary
Ищу: Warhammer 40k: Space Marine game movie no commentary
Ищу: Warhammer 40k: Space Marine walkthrough no commentary
Ищу: Warhammer 40k: Space Mar

Название,Канал,Длительность,Score,Видео
Warhammer 40000 Space Marine 2 ИГРОФИЛЬМ на русском ● PC прохождение без комментариев ● BFGames,BFGames,5h 1m 9s,19,Открыть
Warhammer 40000 Space Marine ИГРОФИЛЬМ на русском ● PC прохождение без комментариев ● BFGames,BFGames,1h 43m 40s,17,Открыть
ИГРОФИЛЬМ Warhammer 40k Space Marine [Прохождение без комментариев] Русская озвучка,Dmitry V00lf,2h 22m 14s,16,Открыть
"WARHAMMER 40,000 SPACE MARINE All Cutscenes (Full Game Movie) 4K UHD",Gamer's Little Playground,1h 59m 41s,14,Открыть
"WARHAMMER 40,000 Space Marine Gameplay Walkthrough Part 1 FULL GAME [4K 60FPS PC] - No Commentary",MKIceAndFire,4h 38m 13s,12,Открыть


Вот, ну совсем другое дело!

Но было бы прикольно еще интерфейс какой-нибудь запилить, думаю.

In [72]:
import gradio as gr
import html

def format_links_markdown(ranked, top_k=5):
    if not ranked:
        return "Подходящих видео не найдено."

    lines = ["## Ссылки на найденные видео", ""]
    for i, item in enumerate(ranked[:top_k], start=1):
        v = item.video
        title = html.escape(v.title)
        channel = html.escape(v.channel_title)
        lines.append(
            f"{i}. [{title}]({v.url})  \n"
            f"   Канал: {channel}  \n"
            f"   Длительность: {format_seconds(v.duration_seconds)}  \n"
            f"   Score: {item.score}"
        )
        lines.append("")
    return "\n".join(lines)


def gradio_agent(user_query):
    user_query = (user_query or "").strip()
    if not user_query:
        return "Введи запрос.", "Нечего показывать."

    try:
        result = run_agent(user_query)
        answer = result["answer"]
        links_md = format_links_markdown(result["ranked"], top_k=5)
        return answer, links_md
    except Exception as e:
        return f"Ошибка: {e}", "Ссылки недоступны."

In [74]:
with gr.Blocks(title="Game Movie Agent") as demo:
    gr.Markdown("# Диалоговый агент поиска игрофильмов без комментариев")
    gr.Markdown(
        "Введи название игры или полный запрос, например:  \n"
        "`Найди игрофильм по Detroit Become Human без комментариев`"
    )

    with gr.Row():
        query_box = gr.Textbox(
            label="Запрос",
            placeholder="Найди игрофильм по Warhammer 40k: Space Marine без комментариев",
            lines=2
        )

    run_btn = gr.Button("Найти")

    answer_box = gr.Textbox(
        label="Ответ агента",
        lines=12
    )

    links_box = gr.Markdown(
        label="Ссылки"
    )

    examples = gr.Examples(
        examples=[
            ["Найди игрофильм по Warhammer 40k: Space Marine без комментариев"],
            ["Хочу полное прохождение Silent Hill 2 без комментариев"],
            ["Найди игрофильм по Detroit Become Human без комментариев"],
        ],
        inputs=query_box
    )

    run_btn.click(
        fn=gradio_agent,
        inputs=query_box,
        outputs=[answer_box, links_box]
    )

    query_box.submit(
        fn=gradio_agent,
        inputs=query_box,
        outputs=[answer_box, links_box]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1e601ee624008d3589.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Смотрим метрику Semantic Similarity через эмбеддинги

In [75]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def semantic_score(reference: str, candidates: list):
    ref_emb = model.encode(reference)

    scores = []
    for text in candidates:
        emb = model.encode(text)
        score = cosine_sim(ref_emb, emb)
        scores.append(score)

    return scores

#эталон
reference = "Warhammer 40k Space Marine full game movie no commentary"

#названия
titles = [item.video.title for item in result["ranked"][:5]]

scores = semantic_score(reference, titles)

for i, (title, score) in enumerate(zip(titles, scores), start=1):
    print(f"{i}. Score: {score:.4f} | {title}")

print(f"\nAverage Semantic@5: {np.mean(scores):.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


1. Score: 0.7502 | Warhammer 40000 Space Marine 2 ИГРОФИЛЬМ на русском ● PC прохождение без комментариев ● BFGames
2. Score: 0.7261 | Warhammer 40000 Space Marine ИГРОФИЛЬМ на русском ● PC прохождение без комментариев ● BFGames
3. Score: 0.6673 | ИГРОФИЛЬМ Warhammer 40k Space Marine [Прохождение без комментариев] Русская озвучка
4. Score: 0.7501 | WARHAMMER 40,000 SPACE MARINE All Cutscenes (Full Game Movie) 4K UHD
5. Score: 0.8019 | WARHAMMER 40,000 Space Marine Gameplay Walkthrough Part 1 FULL GAME [4K 60FPS PC] - No Commentary

Average Semantic@5: 0.7391
